# Search Laboratory: BFS and A*

This notebook implements A* with a replaceable heuristic, BFS as the blind-search
baseline, path reconstruction, state-expansion metrics, the four required correctness
tests, and the heuristic experiments from the assignment.

In [1]:
from collections import deque
from dataclasses import dataclass
from itertools import count
import heapq
import math

ORIGINAL = [
    "#################",
    "#S....#.........#",
    "#.###.#.#######.#",
    "#...#.#.......#.#",
    "###.#.#######.#.#",
    "#...#.........#.#",
    "#.###########.#.#",
    "#.............#G#",
    "#################",
]

DIRECTIONS = (
    ("Up", -1, 0),
    ("Down", 1, 0),
    ("Left", 0, -1),
    ("Right", 0, 1),
)


@dataclass
class SearchResult:
    algorithm: str
    positions: list[tuple[int, int]]
    actions: list[str]
    states_expanded: int

    @property
    def found(self):
        return bool(self.positions)

    @property
    def path_length(self):
        return len(self.actions) if self.found else None


def parse_grid(grid):
    if not grid or len({len(row) for row in grid}) != 1:
        raise ValueError("Grid must be non-empty and rectangular")
    found = {}
    for r, row in enumerate(grid):
        for c, value in enumerate(row):
            if value in "SG":
                if value in found:
                    raise ValueError(f"Grid contains more than one {value}")
                found[value] = (r, c)
    if set(found) != {"S", "G"}:
        raise ValueError("Grid must contain exactly one S and one G")
    return found["S"], found["G"]


def successors(grid, state):
    rows, cols = len(grid), len(grid[0])
    for action, dr, dc in DIRECTIONS:
        nxt = (state[0] + dr, state[1] + dc)
        r, c = nxt
        if 0 <= r < rows and 0 <= c < cols and grid[r][c] != "#":
            yield action, nxt


def reconstruct(parent, goal, algorithm, expanded):
    positions, actions = [], []
    state = goal
    while state is not None:
        positions.append(state)
        previous, action = parent[state]
        if action is not None:
            actions.append(action)
        state = previous
    return SearchResult(
        algorithm,
        list(reversed(positions)),
        list(reversed(actions)),
        expanded,
    )


def bfs(grid):
    start, goal = parse_grid(grid)
    frontier = deque([start])
    parent = {start: (None, None)}
    expanded = 0
    while frontier:
        state = frontier.popleft()
        if state == goal:
            return reconstruct(parent, goal, "BFS", expanded)
        expanded += 1
        for action, nxt in successors(grid, state):
            if nxt not in parent:
                parent[nxt] = (state, action)
                frontier.append(nxt)
    return SearchResult("BFS", [], [], expanded)


def astar(grid, heuristic, name="A*"):
    start, goal = parse_grid(grid)
    tie_breaker = count()
    frontier = [(heuristic(start, goal), next(tie_breaker), 0, start)]
    best_g = {start: 0}
    parent = {start: (None, None)}
    expanded = 0

    while frontier:
        f_score, _, g_score, state = heapq.heappop(frontier)
        if g_score != best_g.get(state):  # stale heap entry
            continue
        if state == goal:
            return reconstruct(parent, goal, name, expanded)
        expanded += 1

        for action, nxt in successors(grid, state):
            candidate_g = g_score + 1
            if candidate_g < best_g.get(nxt, math.inf):
                best_g[nxt] = candidate_g
                parent[nxt] = (state, action)
                candidate_f = candidate_g + heuristic(nxt, goal)
                heapq.heappush(
                    frontier,
                    (candidate_f, next(tie_breaker), candidate_g, nxt),
                )
    return SearchResult(name, [], [], expanded)


def manhattan(state, goal):
    return abs(state[0] - goal[0]) + abs(state[1] - goal[1])


def validate_result(grid, result):
    start, goal = parse_grid(grid)
    if not result.found:
        return False
    assert result.positions[0] == start and result.positions[-1] == goal
    assert len(result.positions) == len(result.actions) + 1
    for previous, current in zip(result.positions, result.positions[1:]):
        assert sum(abs(a - b) for a, b in zip(previous, current)) == 1
        assert grid[current[0]][current[1]] != "#"
    return True

## Required correctness tests

In [2]:
TRIVIAL = [
    "#####",
    "#SG##",
    "#####",
]

NO_SOLUTION = [
    "#######",
    "#S....#",
    "###.###",
    "#...#G#",
    "#######",
]

ALTERNATIVE_PATHS = [
    "#######",
    "#S....#",
    "#.....#",
    "#....G#",
    "#######",
]

original_astar = astar(ORIGINAL, manhattan, "A* Manhattan")
trivial = astar(TRIVIAL, manhattan, "A* trivial")
impossible = astar(NO_SOLUTION, manhattan, "A* no solution")
alternative = astar(ALTERNATIVE_PATHS, manhattan, "A* alternatives")

assert validate_result(ORIGINAL, original_astar)
assert trivial.path_length == 1 and validate_result(TRIVIAL, trivial)
assert not impossible.found
assert validate_result(ALTERNATIVE_PATHS, alternative)
assert alternative.path_length == bfs(ALTERNATIVE_PATHS).path_length

print("Original warehouse")
print("  Found:", original_astar.found)
print("  Path:", original_astar.positions)
print("  Actions:", original_astar.actions)
print("  Path length:", original_astar.path_length)
print("  States expanded:", original_astar.states_expanded)
print("\nTrivial path length:", trivial.path_length)
print("No-solution case found a path:", impossible.found)
print("Alternative-path map shortest length:", alternative.path_length)
print("All four required tests passed.")

Original warehouse
  Found: True
  Path: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (5, 11), (5, 12), (5, 13), (4, 13), (3, 13), (3, 12), (3, 11), (3, 10), (3, 9), (3, 8), (3, 7), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (2, 15), (3, 15), (4, 15), (5, 15), (6, 15), (7, 15)]
  Actions: ['Right', 'Right', 'Right', 'Right', 'Down', 'Down', 'Down', 'Down', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Up', 'Up', 'Left', 'Left', 'Left', 'Left', 'Left', 'Left', 'Up', 'Up', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Down', 'Down', 'Down', 'Down', 'Down', 'Down']
  Path length: 40
  States expanded: 63

Trivial path length: 1
No-solution case found a path: False
Alternative-path map shortest length: 6
All four required tests passed.


## BFS versus A* on the same warehouse

In [3]:
original_bfs = bfs(ORIGINAL)
assert validate_result(ORIGINAL, original_bfs)
assert original_bfs.path_length == original_astar.path_length

print(f"{'Algorithm':<16} {'Found':<8} {'Path length':<14} {'States expanded'}")
for result in (original_bfs, original_astar):
    print(
        f"{result.algorithm:<16} {str(result.found):<8} "
        f"{result.path_length:<14} {result.states_expanded}"
    )

Algorithm        Found    Path length    States expanded
BFS              True     40             63
A* Manhattan     True     40             63


## Heuristic investigation

Manhattan distance is admissible and consistent for unit-cost horizontal/vertical
movement. `h=0` reduces A* to uniform-cost search (equivalent to BFS here). Euclidean
distance is also admissible but usually less informed. `2 * Manhattan` is aggressive
and not admissible; it may expand fewer states, but optimality is no longer guaranteed.

In [4]:
heuristics = [
    ("h = 0", lambda state, goal: 0),
    ("Manhattan", manhattan),
    (
        "Euclidean",
        lambda state, goal: math.hypot(state[0] - goal[0], state[1] - goal[1]),
    ),
    ("2 x Manhattan", lambda state, goal: 2 * manhattan(state, goal)),
]

heuristic_results = [astar(ORIGINAL, fn, name) for name, fn in heuristics]
print(f"{'Heuristic':<18} {'Found':<8} {'Path length':<14} {'States expanded'}")
for result in heuristic_results:
    assert result.found and validate_result(ORIGINAL, result)
    print(
        f"{result.algorithm:<18} {str(result.found):<8} "
        f"{result.path_length:<14} {result.states_expanded}"
    )

Heuristic          Found    Path length    States expanded
h = 0              True     40             63
Manhattan          True     40             63
Euclidean          True     40             63
2 x Manhattan      True     40             66
